# 00 · Descarga de datos y cierres ajustados trimestrales

En este notebook se descargan de Yahoo Finance los precios diarios de cierre ajustado de SPY, LQD y GLD para las dos ventanas que define el enunciado: la de calibración, de diciembre de 2009 a diciembre de 2025, y la de validación, de diciembre de 2025 a septiembre de 2026. De cada serie se conserva solo el último cierre de cada trimestre, se verifica que corresponda al último día de negociación de la bolsa de Nueva York y se guardan los archivos trimestrales con su fuente y fecha de consulta. Aquí no se calculan rendimientos ni estadísticas, eso se hace en el notebook 01, únicamente con los datos de calibración. Los datos de 2026 se descargan y se revisan, pero sus precios no se muestran ni se usan hasta la validación final.

In [1]:
import hashlib
import json
import platform
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd

## 1. Configuración

Aquí se definen los tres activos, siempre en el orden SPY, LQD y GLD, y las fechas de inicio y fin de cada ventana de datos. Como la herramienta de descarga no incluye la fecha final que se le pide, cada ventana se solicita hasta el día siguiente al último que se necesita.

El parámetro `FORCE_DOWNLOAD` protege la reproducibilidad del trabajo. Yahoo Finance recalcula toda la serie de cierres ajustados cada vez que un fondo hace un nuevo pago, de modo que dos descargas hechas en fechas distintas pueden dar niveles diferentes. Por eso, una vez guardados los archivos trimestrales, el notebook los reutiliza y solo vuelve a verificarlos. Para descargar de nuevo hay que cambiarlo a `True` de forma manual.

In [6]:
# ---- Activos (orden fijo SPY, LQD, GLD) ----
TICKERS = ["SPY", "LQD", "GLD"]

# ---- Ventanas de consulta con fin exclusivo  ----
BLOCKS = {
    "calibration": {
        "label": "Calibración",
        "start": "2009-12-01",
        "end_inclusive": "2025-12-31",
        "end_exclusive": "2026-01-01",
        "first_quarter": "2009Q4",   
        "last_quarter": "2025Q4",
        "n_prices": 65,         
        "index_name": "j",         
    },
    "validation": {
        "label": "Validación (reservada)",
        "start": "2025-12-01",
        "end_inclusive": "2026-09-30",
        "end_exclusive": "2026-10-01",
        "first_quarter": "2025Q4",   
        "last_quarter": "2026Q3",
        "n_prices": 4,          
        "index_name": "tau",      
    },
}

# ---- Método de descarga ----
# "yfinance"   : librería yfinance con auto_adjust=False
DOWNLOAD_METHOD = "yfinance"

FORCE_DOWNLOAD = False

# ---- Zona horaria y rutas ----
NY_TZ = ZoneInfo("America/New_York")
BOG_TZ = ZoneInfo("America/Bogota")

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"   
for d in [RAW_DIR, DATA_DIR / "calibration", DATA_DIR / "validation"]:
    d.mkdir(parents=True, exist_ok=True)

def quarterly_csv_path(block):
    tag = {"calibration": "2009_2025", "validation": "2025_2026"}[block]
    return DATA_DIR / block / f"quarterly_adjclose_{tag}.csv"

def metadata_path(block):
    return DATA_DIR / block / "metadata.json"

## 2. Funciones de descarga

Ambas rutas devuelven la misma estructura: un `DataFrame` diario indexado por la **fecha de la sesión en Nueva York**, con el cierre sin ajustar (`close`, solo para documentar el ajuste) y el cierre ajustado (`adj_close`, el único que se usa en el modelo).

- **`yfinance`.** Se fija `auto_adjust=False` para que la librería conserve la columna `Adj Close` en lugar de reemplazar el cierre por un precio ajustado con otra convención, y `actions=False` para no traer dividendos como columnas que luego pudieran sumarse otra vez. Su parámetro `end` es exclusivo.
- **`yahoo_json`.** Lee `timestamp` e `indicators/adjclose` e intenta `query1` y luego `query2`. Los timestamps vienen en segundos UTC y se convierten a la zona horaria de la bolsa antes de tomar la fecha.

In [7]:
def _to_ny_session_dates(index):
    # Convierte un índice de fechas/horas en fechas de sesión de Nueva York (sin hora).
    idx = pd.DatetimeIndex(index)
    if idx.tz is not None:
        idx = idx.tz_convert(NY_TZ).tz_localize(None)
    return pd.DatetimeIndex(idx.normalize(), name="date")


def download_yfinance(ticker, start, end_exclusive):
    import yfinance as yf
    df = yf.download(
        ticker, start=start, end=end_exclusive, interval="1d",
        auto_adjust=False, actions=False, progress=False, threads=False,
    )
    if df is None or df.empty:
        raise RuntimeError(f"yfinance no devolvió datos para {ticker}")
    if isinstance(df.columns, pd.MultiIndex):          # columnas (Price, Ticker)
        df = df.xs(ticker, axis=1, level=-1)
    if "Adj Close" not in df.columns:
        raise RuntimeError(
            "No existe la columna 'Adj Close'. Revisen que auto_adjust=False "
            "esté siendo respetado por su versión de yfinance."
        )
    out = pd.DataFrame(
        {"close": df["Close"].to_numpy(dtype=float),
         "adj_close": df["Adj Close"].to_numpy(dtype=float)},
        index=_to_ny_session_dates(df.index),
    )
    info = {"tool": "yfinance", "tool_version": getattr(yf, "__version__", "desconocida"),
            "call": (f"yf.download('{ticker}', start='{start}', end='{end_exclusive}', "
                     "interval='1d', auto_adjust=False, actions=False)")}
    return out, info


def download_yahoo_json(ticker, start, end_exclusive, override_url=None):
    import requests
    p1 = int(pd.Timestamp(start, tz=NY_TZ).timestamp())
    p2 = int(pd.Timestamp(end_exclusive, tz=NY_TZ).timestamp())
    params = {"period1": p1, "period2": p2, "interval": "1d",
              "events": "div,split", "includeAdjustedClose": "true"}
    headers = {"User-Agent": "Mozilla/5.0"}
    candidates = []
    if override_url:
        candidates.append((override_url, None))
        candidates.append((override_url.replace("query1", "query2"), None))
    for host in ["query1", "query2"]:
        candidates.append((f"https://{host}.finance.yahoo.com/v8/finance/chart/{ticker}", params))

    errors = []
    for url, prm in candidates:
        try:
            r = requests.get(url, params=prm, headers=headers, timeout=30)
            r.raise_for_status()
            res = r.json()["chart"]["result"][0]
            tzname = res.get("meta", {}).get("exchangeTimezoneName", "America/New_York")
            ts = pd.to_datetime(res["timestamp"], unit="s", utc=True).tz_convert(tzname)
            out = pd.DataFrame(
                {"close": np.asarray(res["indicators"]["quote"][0]["close"], dtype=float),
                 "adj_close": np.asarray(res["indicators"]["adjclose"][0]["adjclose"], dtype=float)},
                index=_to_ny_session_dates(ts),
            )
            return out, {"tool": "yahoo_json", "url": r.url, "exchange_timezone": tzname}
        except Exception as e:
            errors.append(f"{url}: {e}")
    raise RuntimeError("Fallaron todas las rutas JSON:\n" + "\n".join(errors))


def download_daily(block, ticker):
    cfg = BLOCKS[block]
    if DOWNLOAD_METHOD == "yfinance":
        return download_yfinance(ticker, cfg["start"], cfg["end_exclusive"])
    if DOWNLOAD_METHOD == "yahoo_json":
        return download_yahoo_json(ticker, cfg["start"], cfg["end_exclusive"],
                                   JSON_URLS_OVERRIDE.get((block, ticker)))
    raise ValueError(f"Método desconocido: {DOWNLOAD_METHOD}")


def clean_daily(df, start, end_inclusive):
    # Limpieza mínima y documentada: orden, duplicados, ventana y valores faltantes.
    rep = {"rows_downloaded": int(len(df))}
    df = df.sort_index()
    dup = df.index.duplicated(keep="last")
    rep["duplicates_removed"] = int(dup.sum())
    df = df[~dup]
    inside = (df.index >= pd.Timestamp(start)) & (df.index <= pd.Timestamp(end_inclusive))
    rep["rows_outside_window_removed"] = int((~inside).sum())
    df = df[inside]
    nan = df["adj_close"].isna()
    rep["rows_nan_adj_close_removed"] = int(nan.sum())
    df = df[~nan]
    if (df["adj_close"] <= 0).any():
        raise ValueError("Hay cierres ajustados no positivos.")
    rep["rows_kept"] = int(len(df))
    rep["first_session"] = df.index.min().date().isoformat()
    rep["last_session"] = df.index.max().date().isoformat()
    return df, rep

## 3. Último día de negociación de cada trimestre y verificaciones

Para cada trimestre se necesita el precio del último día en que la bolsa abrió. Normalmente es el último día del mes, pero si cae en fin de semana o festivo hay que usar la jornada anterior. En lugar de suponerlo, el notebook toma la última fecha que aparece en los datos y la compara con el calendario oficial de la bolsa de Nueva York (`pandas_market_calendars`, código `XNYS`). Así se detectan dos errores que invalidarían los resultados: que falte el cierre de un trimestre y se termine usando un precio más viejo, o que se trate como completo un trimestre que todavía no ha terminado. En cualquiera de los dos casos el notebook se detiene.

Si `pandas_market_calendars` no está instalado, se aplica una revisión más simple, que solo exige un día hábil dentro de la última semana del trimestre, y esa limitación queda anotada en la metadata.

In [8]:
def expected_quarter_end_sessions(first_q, last_q):
    # Última sesión oficial de NYSE en cada trimestre del rango.
    try:
        import pandas_market_calendars as mcal
    except ImportError:
        return None, "sin calendario oficial (pandas_market_calendars no instalado)"
    q0, q1 = pd.Period(first_q, "Q"), pd.Period(last_q, "Q")
    sched = mcal.get_calendar("XNYS").schedule(
        start_date=q0.start_time.date(), end_date=q1.end_time.date())
    sessions = pd.DatetimeIndex(sched.index).normalize()
    if sessions.tz is not None:
        sessions = sessions.tz_localize(None)
    s = pd.Series(sessions, index=sessions.to_period("Q"))
    return s.groupby(level=0).max(), "pandas_market_calendars (XNYS)"


EXPECTED_SESSIONS, CALENDAR_METHOD = expected_quarter_end_sessions("2009Q4", "2026Q3")
print("Verificación de calendario:", CALENDAR_METHOD)


def extract_quarter_end(adj, block):
    # adj: DataFrame diario con una columna por activo (cierre ajustado).
    cfg = BLOCKS[block]
    quarters = pd.period_range(cfg["first_quarter"], cfg["last_quarter"], freq="Q")
    rows = []
    for q in quarters:
        last_dates = {}
        for t in TICKERS:
            s = adj[t].dropna()
            sq = s[(s.index >= q.start_time) & (s.index <= q.end_time)]
            if sq.empty:
                raise ValueError(f"{t}: no hay datos en {q}.")
            last_dates[t] = sq.index.max()
        if len(set(last_dates.values())) != 1:
            raise ValueError(f"{q}: los activos terminan en fechas distintas {last_dates}. "
                             "No se reemplaza un cierre faltante por uno anterior.")
        d = last_dates[TICKERS[0]]
        rows.append({"quarter": str(q), "date": d.date().isoformat(),
                     **{t: float(adj.at[d, t]) for t in TICKERS}})
    out = pd.DataFrame(rows)
    out.insert(0, cfg["index_name"], range(len(out)))
    return out


def verify_quarterly(table, block):
    # Verificaciones exigidas por las secciones 2.2 y 2.3. Lanza error si alguna falla.
    cfg = BLOCKS[block]
    checks = []
    quarters = [str(q) for q in pd.period_range(cfg["first_quarter"], cfg["last_quarter"], freq="Q")]

    assert len(table) == cfg["n_prices"], f"Se esperaban {cfg['n_prices']} precios por activo y hay {len(table)}."
    checks.append(f"{cfg['n_prices']} precios por activo")

    assert table["quarter"].tolist() == quarters, "La lista de trimestres no coincide con la ventana."
    checks.append(f"trimestres exactos {quarters[0]} a {quarters[-1]}, sin huecos ni repetidos")

    assert table[cfg["index_name"]].tolist() == list(range(cfg["n_prices"]))
    vals = table[TICKERS].to_numpy(dtype=float)
    assert np.isfinite(vals).all() and (vals > 0).all(), "Hay precios faltantes o no positivos."
    checks.append("precios finitos y positivos")

    dates = pd.to_datetime(table["date"])
    assert dates.is_monotonic_increasing and dates.is_unique
    assert all(d.to_period("Q") == pd.Period(q, "Q") for d, q in zip(dates, table["quarter"]))
    checks.append("cada fecha pertenece a su trimestre, en orden cronológico")

    today_ny = pd.Timestamp(datetime.now(NY_TZ).date())
    if EXPECTED_SESSIONS is not None:
        for d, q in zip(dates, table["quarter"]):
            exp = EXPECTED_SESSIONS.loc[pd.Period(q, "Q")]
            assert d == exp, (f"{q}: el último dato es {d.date()} y la última sesión oficial de NYSE "
                              f"es {exp.date()}. No se sustituye por una cotización anterior.")
        last_exp = EXPECTED_SESSIONS.loc[pd.Period(cfg["last_quarter"], "Q")]
        assert last_exp < today_ny, (f"El trimestre {cfg['last_quarter']} termina el {last_exp.date()} "
                                     "y aún no es un trimestre completo.")
        checks.append("cada fecha es la última sesión oficial NYSE del trimestre (festivos incluidos)")
    else:
        for d, q in zip(dates, table["quarter"]):
            qend = pd.Period(q, "Q").end_time.normalize()
            assert d.weekday() < 5 and (qend - d).days <= 6, f"{q}: fecha final sospechosa {d.date()}."
        assert pd.Period(cfg["last_quarter"], "Q").end_time.normalize() < today_ny
        checks.append("verificación débil de fechas (sin calendario oficial)")
    checks.append(f"último trimestre {cfg['last_quarter']} completo a la fecha de ejecución")
    return checks


def sha256_of(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

Verificación de calendario: pandas_market_calendars (XNYS)


## 4. Función que procesa un bloque

Para cada bloque se descargan los tres activos con la misma función y configuración, se guardan los diarios crudos en `data/raw/` (CSV, no Parquet), se extraen los cierres trimestrales, se verifican y se guardan con su metadata. Si los archivos ya existen y `FORCE_DOWNLOAD = False`, se cargan y se vuelven a verificar sin descargar.

In [9]:
def build_block(block):
    cfg = BLOCKS[block]
    csv_path, meta_path = quarterly_csv_path(block), metadata_path(block)

    if csv_path.exists() and meta_path.exists() and not FORCE_DOWNLOAD:
        table = pd.read_csv(csv_path, dtype={"quarter": str, "date": str})
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        assert sha256_of(csv_path) == meta["csv_sha256"], (
            f"{csv_path.name} cambió después de guardarse (hash distinto).")
        checks = verify_quarterly(table, block)
        print(f"[{cfg['label']}] archivos existentes reutilizados (consulta original: "
              f"{meta['consulted_at_bogota']}).")
        return table, meta, checks

    consulted_utc = datetime.now(timezone.utc)
    series, cleaning, sources = {}, {}, {}
    for t in TICKERS:
        raw, info = download_daily(block, t)
        raw.to_csv(RAW_DIR / f"{block}_{t}_daily.csv", index_label="date")
        clean, rep = clean_daily(raw, cfg["start"], cfg["end_inclusive"])
        series[t], cleaning[t], sources[t] = clean["adj_close"], rep, info
        print(f"[{cfg['label']}] {t}: {rep['rows_kept']} sesiones "
              f"({rep['first_session']} a {rep['last_session']}); "
              f"filas sin Adj Close eliminadas: {rep['rows_nan_adj_close_removed']}")

    adj = pd.concat(series, axis=1)[TICKERS]
    table = extract_quarter_end(adj, block)
    checks = verify_quarterly(table, block)

    table.to_csv(csv_path, index=False, float_format="%.10f")
    meta = {
        "block": block,
        "purpose": ("Estimar probabilidades y factores; construir el árbol" if block == "calibration"
                    else "Validación fuera de muestra; no se usa para estimar ni ajustar la política"),
        "source": "Yahoo Finance (históricos diarios)",
        "field": "Adj Close (cierre ajustado, en USD)",
        "download_method": DOWNLOAD_METHOD,
        "auto_adjust": False,
        "dividends_added_separately": False,
        "requested_window": {"start": cfg["start"], "end_inclusive": cfg["end_inclusive"],
                             "end_exclusive_sent_to_tool": cfg["end_exclusive"]},
        "date_convention": "Fecha de la sesión en America/New_York",
        "quarter_end_rule": "Última sesión de negociación del trimestre calendario",
        "calendar_check": CALENDAR_METHOD,
        "consulted_at_utc": consulted_utc.isoformat(timespec="seconds"),
        "consulted_at_bogota": consulted_utc.astimezone(BOG_TZ).isoformat(timespec="seconds"),
        "tickers": TICKERS,
        "sources_by_ticker": sources,
        "daily_cleaning_by_ticker": cleaning,
        "n_prices_per_asset": cfg["n_prices"],
        "quarters": [table["quarter"].iloc[0], table["quarter"].iloc[-1]],
        "index_name": cfg["index_name"],
        "csv_file": str(csv_path.relative_to(ROOT)),
        "csv_sha256": sha256_of(csv_path),
        "checks_passed": checks,
        "environment": {"python": platform.python_version(), "pandas": pd.__version__,
                        "numpy": np.__version__},
    }
    meta_path.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"[{cfg['label']}] guardado: {csv_path.relative_to(ROOT)} y {meta_path.relative_to(ROOT)}")
    return table, meta, checks

## 5. Bloque de calibración (dic-2009 a dic-2025)

Se esperan 65 cierres ajustados por activo: diciembre de 2009, que es solo el punto de partida $P_{i,0}$, y el cierre de cada trimestre de 2010 a 2025, $P_{i,1},\dots,P_{i,64}$. 

In [10]:
cal_q, cal_meta, cal_checks = build_block("calibration")
for c in cal_checks:
    print("  ✔", c)

[Calibración] archivos existentes reutilizados (consulta original: 2026-10-04T16:26:02-05:00).
  ✔ 65 precios por activo
  ✔ trimestres exactos 2009Q4 a 2025Q4, sin huecos ni repetidos
  ✔ precios finitos y positivos
  ✔ cada fecha pertenece a su trimestre, en orden cronológico
  ✔ cada fecha es la última sesión oficial NYSE del trimestre (festivos incluidos)
  ✔ último trimestre 2025Q4 completo a la fecha de ejecución


In [11]:
with pd.option_context("display.float_format", "{:.4f}".format):
    display(pd.concat([cal_q.head(5), cal_q.tail(5)]))

,j,quarter,date,SPY,LQD,GLD
0,0,2009Q4,2009-12-31,82.9620,56.2055,107.3100
1,1,2010Q1,2010-03-31,87.4598,57.5682,108.9500
2,2,2010Q2,2010-06-30,77.5261,59.7971,121.6800
3,3,2010Q3,2010-09-30,86.1792,63.0895,127.9100
4,4,2010Q4,2010-12-31,95.4529,61.4418,138.7200
60,60,2024Q4,2024-12-31,574.7880,98.5893,242.1300
61,61,2025Q1,2025-03-31,550.2625,101.0434,288.1400
62,62,2025Q2,2025-06-30,609.5654,103.0807,304.8300
63,63,2025Q3,2025-09-30,659.0695,106.0165,355.4700
64,64,2025Q4,2025-12-31,676.6350,106.3733,396.3100


## 6. Bloque de validación (dic-2025 a sep-2026)
Se esperan cuatro cierres ajustados por activo: diciembre de 2025 ($\tilde P_{i,0}$) y marzo, junio y septiembre de 2026 ($\tilde P_{i,1}, \tilde P_{i,2}, \tilde P_{i,3}$). La verificación exige que el cierre de septiembre sea la última sesión oficial del trimestre y que el trimestre ya haya terminado.

Solo se muestran fechas y trimestres, no precios. Estos datos se leen de nuevo únicamente en el notebook 06, después de guardar la política.

In [12]:
val_q, val_meta, val_checks = build_block("validation")
for c in val_checks:
    print("  ✔", c)

# Solo información estructural; los precios de 2026 no se imprimen.
display(val_q[["tau", "quarter", "date"]])

[Validación (reservada)] archivos existentes reutilizados (consulta original: 2026-10-04T16:26:16-05:00).
  ✔ 4 precios por activo
  ✔ trimestres exactos 2025Q4 a 2026Q3, sin huecos ni repetidos
  ✔ precios finitos y positivos
  ✔ cada fecha pertenece a su trimestre, en orden cronológico
  ✔ cada fecha es la última sesión oficial NYSE del trimestre (festivos incluidos)
  ✔ último trimestre 2026Q3 completo a la fecha de ejecución


,tau,quarter,date
0,0,2025Q4,2025-12-31
1,1,2026Q1,2026-03-31
2,2,2026Q2,2026-06-30
3,3,2026Q3,2026-09-30


## 7. Misma definición de precio en ambos bloques

Los datos de calibración y los de validación tienen que ser el mismo tipo de precio, y los pagos de los fondos no deben contarse dos veces. Para comprobarlo se revisan dos cosas.

La primera es el cierre de diciembre de 2025, que aparece en los dos bloques y debe corresponder al mismo día. Si ambos se descargaron en la misma sesión, el precio también debe coincidir. Si se descargaron en días distintos, puede diferir por un factor constante, porque Yahoo reescala la serie ajustada cada vez que un fondo hace un pago. Eso no afecta los rendimientos, ya que se calculan como cocientes de precios del mismo bloque y el reescalamiento multiplica todos esos precios por igual, pero queda registrado.

La segunda es una prueba de que el ajuste está incluido. Con los datos diarios se compara el cierre ajustado con el cierre normal al inicio y al final de la ventana de calibración. Si al inicio el cociente es menor que uno, la serie ya incorpora los pagos reinvertidos y no hace falta sumarlos aparte.

In [13]:
c25 = cal_q.set_index("quarter").loc["2025Q4"]
v25 = val_q.set_index("quarter").loc["2025Q4"]
assert c25["date"] == v25["date"], "El cierre de dic-2025 no corresponde a la misma sesión en ambos bloques."
print(f"Sesión de dic-2025 en ambos bloques: {c25['date']}")

ratios = {t: float(v25[t]) / float(c25[t]) for t in TICKERS}
for t, r in ratios.items():
    print(f"  {t}: nivel validación / nivel calibración = {r:.10f}")
if all(abs(r - 1) < 1e-8 for r in ratios.values()):
    print("Los niveles coinciden: misma serie ajustada en ambos bloques.")
else:
    print("Los niveles difieren por un factor de escala (causa habitual: consultas en fechas distintas "
          "o un pago registrado entre ellas). "
          "Los rendimientos de cada bloque se calculan solo con precios de ese bloque, así que "
          "el factor no los altera; quede documentado en el informe.")

rows = []
for t in TICKERS:
    p = RAW_DIR / f"calibration_{t}_daily.csv"
    if not p.exists():
        print(f"(Sin diario crudo de {t}; se omite la evidencia del ajuste.)")
        continue
    d = pd.read_csv(p, parse_dates=["date"], index_col="date").dropna()
    d = d.loc[BLOCKS["calibration"]["start"]:BLOCKS["calibration"]["end_inclusive"]]
    rows.append({"activo": t,
                 "Adj/Close primera sesión": d["adj_close"].iloc[0] / d["close"].iloc[0],
                 "Adj/Close última sesión": d["adj_close"].iloc[-1] / d["close"].iloc[-1]})
if rows:
    with pd.option_context("display.float_format", "{:.6f}".format):
        display(pd.DataFrame(rows).set_index("activo"))

Sesión de dic-2025 en ambos bloques: 2025-12-31
  SPY: nivel validación / nivel calibración = 1.0000000000
  LQD: nivel validación / nivel calibración = 0.9999999283
  GLD: nivel validación / nivel calibración = 1.0000000000
Los niveles difieren por un factor de escala (causa habitual: consultas en fechas distintas o un pago registrado entre ellas). Los rendimientos de cada bloque se calculan solo con precios de ese bloque, así que el factor no los altera; quede documentado en el informe.


,Adj/Close primera sesión,Adj/Close última sesión
activo,,
SPY,0.740468,0.992250
LQD,0.537408,0.965363
GLD,1.000000,1.000000


## 8. Resumen y procedimiento para el informe

La siguiente celda resume los archivos producidos y redacta el procedimiento de obtención de datos con los valores reales de esta ejecución, listo para adaptarlo en el informe.

In [14]:
for name, meta in [("calibration", cal_meta), ("validation", val_meta)]:
    print(f"{BLOCKS[name]['label']}")
    print(f"  archivo     : {meta['csv_file']}")
    print(f"  SHA-256     : {meta['csv_sha256']}")
    print(f"  consulta    : {meta['consulted_at_bogota']} (hora de Bogotá)")
    print(f"  trimestres  : {meta['quarters'][0]} a {meta['quarters'][1]} "
          f"({meta['n_prices_per_asset']} precios por activo)")

m = cal_meta
tool = m["sources_by_ticker"][TICKERS[0]]
tool_txt = (f"la librería yfinance (versión {tool.get('tool_version')}) con auto_adjust=False"
            if m["download_method"] == "yfinance"
            else "los enlaces JSON del proveedor (campo indicators/adjclose)")
print(
    f"Los precios se obtuvieron de Yahoo Finance mediante {tool_txt}, para SPY, LQD y GLD con "
    f"frecuencia diaria. Se conservó únicamente la fecha y el cierre ajustado (Adj Close), sin "
    f"agregar dividendos por separado. El bloque de calibración se solicitó del "
    f"{BLOCKS['calibration']['start']} al {BLOCKS['calibration']['end_inclusive']} (fin exclusivo "
    f"{BLOCKS['calibration']['end_exclusive']}) y el de validación del {BLOCKS['validation']['start']} "
    f"al {BLOCKS['validation']['end_inclusive']} (fin exclusivo {BLOCKS['validation']['end_exclusive']}). "
    f"Las fechas se expresaron en el calendario bursátil de Nueva York y, para cada trimestre "
    f"calendario, se tomó el cierre ajustado de la última sesión, contrastada con el calendario "
    f"oficial de NYSE ({m['calendar_check']}). Se obtuvieron 65 precios por activo para calibración "
    f"y 4 para validación. Las consultas se realizaron el {cal_meta['consulted_at_bogota'][:10]} "
    f"(calibración) y el {val_meta['consulted_at_bogota'][:10]} (validación)."
)

Calibración
  archivo     : data/calibration/quarterly_adjclose_2009_2025.csv
  SHA-256     : a6ae2e777012912a7e1385863c1ed83f42de937ac72b92c0cdee930186a239dc
  consulta    : 2026-10-04T16:26:02-05:00 (hora de Bogotá)
  trimestres  : 2009Q4 a 2025Q4 (65 precios por activo)
Validación (reservada)
  archivo     : data/validation/quarterly_adjclose_2025_2026.csv
  SHA-256     : de49ef8063f0c97d532a2bd7f44454d153dd2248ae94b0360c02bd70b0f5a645
  consulta    : 2026-10-04T16:26:16-05:00 (hora de Bogotá)
  trimestres  : 2025Q4 a 2026Q3 (4 precios por activo)
Los precios se obtuvieron de Yahoo Finance mediante la librería yfinance (versión 1.7.0) con auto_adjust=False, para SPY, LQD y GLD con frecuencia diaria. Se conservó únicamente la fecha y el cierre ajustado (Adj Close), sin agregar dividendos por separado. El bloque de calibración se solicitó del 2009-12-01 al 2025-12-31 (fin exclusivo 2026-01-01) y el de validación del 2025-12-01 al 2026-09-30 (fin exclusivo 2026-10-01). Las fechas se ex